### Tables

In [0]:
tables = [
    "end_to_end_retail.db_landing.tbl_customer_autoloader",
    "end_to_end_retail.db_landing.tbl_customer_taxid",
    "end_to_end_retail.db_landing.tbl_sales",
    "end_to_end_retail.db_landing.tbl_sales_oders_ordered_product",
    "end_to_end_retail.db_landing.tbl_source_files_customers",
    "end_to_end_retail.db_landing.tbl_source_files_sales",
    "end_to_end_retail.db_landing.tbl_source_files_sales_orders",
    "end_to_end_retail.db_bronze.tbb_bright_home_orders_enriched",
    "end_to_end_retail.db_bronze.tbb_customer_changes_daily",
    "end_to_end_retail.db_bronze.tbb_northstar_outfitters_orders"
]

### UC-Table-ACL

#### Column Access Control Example

In [0]:
%sql
CREATE OR REPLACE FUNCTION end_to_end_retail.db_landing.pii_mask(column_value STRING)
RETURNS STRING
RETURN CASE
  WHEN current_user() = 'user_one@company.com' THEN column_value
  ELSE 'REDACTED'
END;


-- CREATE OR REPLACE FUNCTION main_catalog.default.pii_masking_policy(column_value STRING)
-- RETURN CASE
--   -- Check if the current user is the authorized PII viewer
--   WHEN is_account_group_member('pii_admins') OR current_user() = 'user3_authorized@company.com' 
--   THEN column_value
--   ELSE '***REDACTED***'
-- END;

In [0]:
%sql
-- Cleanup any row-filters or masks that may have been added in previous runs of the demo
-- ALTER TABLE customers DROP ROW FILTER;
-- ALTER TABLE customers ALTER COLUMN address DROP MASK;

In [0]:
%sql
SELECT 
  customer_id,
  end_to_end_retail.db_landing.pii_mask(email) AS masked_email
FROM end_to_end_retail.db_landing.tbl_customer_autoloader
limit 5

#### Row-level access control

In [0]:
%sql
-- get the current user (for informational purposes)
SELECT current_user(), is_account_group_member('account_example');

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.tbl_customer_autoloader limit 10

In [0]:
%sql
CREATE OR REPLACE FUNCTION region_filter(region_param STRING) 
RETURN 
  is_account_group_member('account_example') or  -- bu_admin can access all regions
  region_param like "NY%";                -- non bu_admin's can only access regions containing US

-- Grant access to all users to the function for the demo by making all account users owners.  Note: Don't do this in production!
-- GRANT ALL PRIVILEGES ON FUNCTION region_filter TO `account users`; 



In [0]:
%sql
-- Let's try our filter. As expected, we can access USA but not SPAIN.
SELECT region_filter('NY'), region_filter('GA')

In [0]:
%sql
ALTER TABLE end_to_end_retail.db_landing.tbl_customer_autoloader SET ROW FILTER region_filter ON (state);

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.tbl_customer_autoloader limit 15

In [0]:
%sql
SELECT DISTINCT(state) FROM end_to_end_retail.db_landing.tbl_customer_autoloader;

In [0]:
%sql
ALTER TABLE end_to_end_retail.db_landing.tbl_customer_autoloader DROP ROW FILTER;
-- Confirming that we can once again see all countries:
SELECT DISTINCT(state) FROM end_to_end_retail.db_landing.tbl_customer_autoloader;

#### Column-level access control

#### Masking function (PII mask)

In [0]:
%sql
-- create a SQL function for a simple column mask:
CREATE OR REPLACE FUNCTION end_to_end_retail.db_landing.simple_mask(column_value STRING)
RETURN 
  case when is_account_group_member('group_example') then column_value else  "****" END;
   
-- GRANT ALL PRIVILEGES ON FUNCTION simple_mask TO `account users`; --only for demo, don't do that in prod as everybody could change the function

In [0]:
%sql
-- Updating the existing simple_mask function to provide more advanced masking options via the built-in SQL MASK function:
CREATE OR REPLACE FUNCTION end_to_end_retail.db_landing.simple_mask(column_value STRING)
   RETURN 
      IF(is_account_group_member('group_example'), column_value, MASK(column_value, '*', '*'));
      -- You can also create custom mask transformations, such as concat(substr(maskable_param, 0, 2), "..."))
   
--  -- grant access to all user to the function for the demo  
-- ALTER FUNCTION simple_mask OWNER TO `account users`;

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.tbl_customer_autoloader limit 5

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE end_to_end_retail.db_bronze.tbl_customer_autoloader_masked (
    address STRING,
    city STRING,
    customer_id LONG,
    email STRING MASK end_to_end_retail.db_landing.simple_mask,
    name STRING,
    operation STRING,
    state STRING,
    timestamp DATE,
    zip_code STRING,
    _rescued_data STRING,
    file_path STRING,
    ingestion_date TIMESTAMP
)
""")

In [0]:
spark.sql("""
INSERT INTO end_to_end_retail.db_bronze.tbl_customer_autoloader_masked
SELECT address, city, customer_id, email, name, operation, state, timestamp, zip_code, _rescued_data, file_path, ingestion_date
FROM end_to_end_retail.db_landing.tbl_customer_autoloader
""")

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_bronze.tbl_customer_autoloader_masked limit 100

#### Workings

In [0]:
%sql
CREATE TABLE IF NOT EXISTS 
  map_country_group (
    identity_group STRING,
    countries ARRAY<STRING>
);

-- GRANT ALL PRIVILEGES ON TABLE map_country_group TO `account users`; --only for demo, don't do that in prod as everybody could change the groups

INSERT OVERWRITE 
  map_country_group (identity_group, countries) VALUES
    ('ANALYST_FR', Array("FR", "BELGIUM","CANADA","SWITZERLAND")),
    ('ANALYST_SPAIN',  Array("SPAIN","MEXICO","ARGENTINA")),
    ('ANALYST_USA', Array("USA","CANADA"));

SELECT * FROM map_country_group;

In [0]:
%sql
-- Query the map_country_group table to see how current user is mapped.
-- This should return the rows for ANALYST_USA since our user is assigned to these groups:
SELECT * FROM map_country_group WHERE is_account_group_member(identity_group); 

In [0]:
%sql
-- Create a SQL function for row-level filter based on mapping table lookup:
CREATE OR REPLACE FUNCTION region_filter_dynamic(region_param STRING)
  RETURN 
    is_account_group_member('group_example') or
    exists (  
      SELECT 1 
      FROM map_country_group 
      WHERE is_account_group_member(identity_group) AND array_contains(countries, region_param)
    );

-- GRANT ALL PRIVILEGES ON FUNCTION region_filter_dynamic TO `account users`;

In [0]:
# Identify the Catalog and the Masking Function to use
catalog_name = "main_catalog"
mask_function = "main_catalog.default.pii_masking_policy"
target_tag = "PII"

# Query the Information Schema to find all columns with the specific tag
tagged_columns = spark.sql(f"""
    SELECT table_catalog, table_schema, table_name, column_name
    FROM {catalog_name}.information_schema.column_tags
    WHERE tag_name = '{target_tag}'
""")

for row in tagged_columns.collect():
    full_table_path = f"{row.table_catalog}.{row.table_schema}.{row.table_name}"
    column_name = row.column_name
    
    print(f"Applying mask to {full_table_path}.{column_name}")
    
    # Execute the SQL to set the mask
    spark.sql(f"""
        ALTER TABLE {full_table_path} 
        ALTER COLUMN {column_name} SET MASK {mask_function}
    """)
